# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

In [5]:
Understand the transcriptomic effects of chronic oxycodone exposure and then withdrawal in the presence or absence of chronic neuropathic pain in the brain reward circuitry using a mouse model. Investigate the effect of HDAC1/HDAC2 inhibitor on the transcriptomic effects of withdrawa using bioinformatic predictions from RNA-seq analyses.

SyntaxError: invalid syntax (1682300831.py, line 1)

What do the conditions mean?

oxy: oxycodone


sal: saline

What do the genotypes mean?

SNI:mice with prolonged spared nerve injury = chronic pain states 


Sham:Control

Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

What would you do? QC normaliztaion differential abundance analysis

Which groups would you compare to each other? oxy to sal and  if  sham different behavior

Please also mention which outcome you would expect to see from each comparison. 

In [ ]:
%pip install pandas openpyxl



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [2]:
import panda as pd

ModuleNotFoundError: No module named 'panda'

In [3]:
import sys
print (sys.executable)

C:\Users\Dell\miniconda3_2Install\envs\biomedworkflow-course\python.exe


In [4]:
import sys
!{sys.executable} -m pip install pandas openpyxl

In [5]:
import pandas as pd 
print ("Läuft")

Läuft


Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition? 8
2. How many samples do you have per genotype?  8
3. How often do you have each condition per genotype?

In [6]:
df = pd.read_excel("conditions_runs_oxy_project.xlsx")
df.head()

,Patient,Run,RNA-seq,DNA-seq,condition: Sal,Condition: Oxy,Genotype: SNI,Genotype: Sham
0,?,SRR23195505,x,NaN,x,NaN,x,NaN
1,?,SRR23195506,x,NaN,NaN,x,NaN,x
2,?,SRR23195507,x,NaN,x,NaN,NaN,x
3,?,SRR23195508,x,NaN,NaN,x,x,NaN
4,?,SRR23195509,x,NaN,NaN,x,x,NaN


In [7]:
print(df.columns.tolist())


['Patient', 'Run', 'RNA-seq', 'DNA-seq', 'condition: Sal', 'Condition: Oxy', 'Genotype: SNI', 'Genotype: Sham']


They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
 Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [9]:
basecounts_df = pd.read_csv("base_counts.csv", sep='\t')
display(basecounts_df.head())
print(basecounts_df.columns)

,"Run,Bases"
0,"SRR23195505,6922564500"
1,"SRR23195506,7859530800"
2,"SRR23195507,8063298900"
3,"SRR23195508,6927786900"
4,"SRR23195509,7003550100"


Index(['Run,Bases'], dtype='str')


In [14]:
basecounts_df = pd.read_csv("base_counts.csv")
df_merged = pd.merge(df, basecounts_df, on='Run', how='left')

display(df_merged.head())
df_sorted = df_merged.sort_values(by=['Bases', 'condition: Sal', 'Condition: Oxy'], ascending=[False, True, True])

,Run,RNA-seq,DNA-seq,condition: Sal,Condition: Oxy,Genotype: SNI,Genotype: Sham,Bases
0,SRR23195505,x,NaN,x,NaN,x,NaN,6922564500
1,SRR23195506,x,NaN,NaN,x,NaN,x,7859530800
2,SRR23195507,x,NaN,x,NaN,NaN,x,8063298900
3,SRR23195508,x,NaN,NaN,x,x,NaN,6927786900
4,SRR23195509,x,NaN,NaN,x,x,NaN,7003550100


In [15]:
smallest_runs = df_sorted.sort_values(by='Bases', ascending=True).head(2)
print(smallest_runs['Run'])

11    SRR23195516
6     SRR23195511
Name: Run, dtype: str


In [16]:
smallest_runs[['Run']].to_csv("ids.csv", index=False, header=True)

display(smallest_runs['Run'])

11    SRR23195516
6     SRR23195511
Name: Run, dtype: str

In [18]:
%conda install -c bioconda nextflow -y



Note: you may need to restart the kernel to use updated packages.


Der Befehl "conda" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.